# Section 4.1 — Binary Walking Detection: Supervising Trajectories

Produces, for the binary walking-detection dataset:

* **Table 4** — Cov(X,T), error, and learned couplings for the three
  supervising trajectories
* **Figure** `teacher_error_binary.pdf` — cumulative error, four curves
* **Figure** `teacher_couplings_binary.pdf` — learned couplings vs k

Run the cells in order. Nothing else is needed.

---

### What was wrong in the previous version

1. **`cov_XT` was missing a term.** It computed `mean(X*T)`, which is
   E[XT], not Cov(X,T) = E[XT] − E[X]E[T]. For the annotated arm this
   returned 1.0000 instead of 0.9313, so every value in the comparison
   table was wrong.
2. **The KL panel used the simulation's reference values**
   (`c_ref = 1.0`, `var_ref = 1/3`). Chapter 3 states that a divergence
   computed against the wrong reference *increases as the estimate
   converges correctly*. KL is removed here — there is no calibration
   reference on hardware to diverge from.
3. **The fused arm silently fell back to ground truth** when `k < K0`.
   With K0 = 0 this never fired, but it meant the "fused" arm was one
   variable away from being partly ground-truth taught. Removed.
4. **The Kalman loop was written out twice**, once in each cell, with
   small differences between them. There is now one function.


In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set(style="whitegrid")
sns.set_palette("deep")

CSV = "../data/binary_walking_1851.csv"

# Filter configuration, Table 11
SIG_E2 = 1.0      # expert noise variance
C_E    = 1.0      # expert coupling
Q_VAL  = 10.0
P0_VAL = 1.0
X0_VAL = 0.0
EPS    = 1e-6

# Offline affine references, Table 3 (binary rows)
C_REF     = {1: 4.332, 2: 6.360}
SIGMA_REF = {1: 11.366, 2: 7.357}


## 1. Load and build ground truth

Ground truth follows Eq. (3.9): every mouse press toggles the state, so
the annotated state is the running parity. Presses in adjacent ticks are
de-duplicated first — a press held across a tick boundary would toggle
twice and invert the rest of the recording.

The two assertions are the ones worth keeping. If either fires, stop:
every number downstream depends on them.


In [2]:
df = pd.read_csv(CSV)
df = df[["human", "mouse", "dBFS_mic1", "dBFS_mic2"]].dropna().reset_index(drop=True)

m = df["mouse"].astype(int)
m = m.mask((m == 1) & (m.shift(1, fill_value=0) == 1), 0)   # de-duplicate
walking_gt = (m.cumsum() % 2).astype(int)

N  = len(df)
X  = np.where(walking_gt.to_numpy() == 1, 1, -1)            # annotated state, +-1
y_e = np.where(df["human"].astype(int).to_numpy() == 1, 1.0, -1.0)   # camera

# microphones, mean-centred (Section 3.4; full-recording mean)
y1 = df["dBFS_mic1"].astype(float).to_numpy()
y1 = y1 - y1.mean()
y2 = df["dBFS_mic2"].astype(float).to_numpy()
y2 = y2 - y2.mean()

n_neg, n_pos = int((X == -1).sum()), int((X == 1).sum())
print(f"N = {N}                    (Table 1 expects 1851)")
print(f"class counts = {n_neg} / {n_pos}   (Table 1 expects 1168 / 683)")

assert N == 1851, "row count does not match Table 1 -- stop here"
assert (n_neg, n_pos) == (1168, 683), "class counts do not match -- stop here"


N = 1851                    (Table 1 expects 1851)
class counts = 1168 / 683   (Table 1 expects 1168 / 683)


## 2. The filter

One function, three arms. The only thing that changes between arms is
`teacher_mode`, which selects the signal $T(\cdot)$ used in
Eqs. (3.13)–(3.14).

* `"gt"` — the annotated state. Exact, but unavailable in deployment.
* `"expert"` — the expert-only Kalman filter's decision. Available in
  deployment, and independent of the microphones.
* `"fused"` — the full network's own decision. The framework's choice;
  closes the self-referential loop.

There is no burn-in and no fallback. The fused arm uses the fused
estimate from k = 0, so nothing in it is contaminated by ground truth.


In [3]:
def run(teacher_mode, learn=True, c_init=1.0, s_init=1.0):
    """Expert + two inexpert sensors. Returns decisions and parameter histories."""
    A = np.eye(1)
    Q = Q_VAL * np.eye(1)

    Xf, Pf = np.array([X0_VAL]), np.eye(1) * P0_VAL       # fused filter
    Xe, Pe = np.array([X0_VAL]), np.eye(1) * P0_VAL       # expert-only filter

    c1h = c2h = float(c_init)
    s1h = s2h = float(s_init)

    dec     = np.zeros(N)
    exp_dec = np.zeros(N)
    hist    = {k: np.zeros(N) for k in ("c1", "c2", "s1", "s2")}

    C_e = np.array([[C_E]])
    R_e = np.array([[SIG_E2]])

    for k in range(N):
        # ---- expert-only filter (never sees the microphones) ----
        Xm = A @ Xe.reshape(-1, 1)
        Pm = A @ Pe @ A.T + Q
        K  = Pm @ C_e.T @ np.linalg.inv(C_e @ Pm @ C_e.T + R_e)
        Xe = (Xm + K @ (y_e[k] - C_e @ Xm)).flatten()
        Pe = (np.eye(1) - K @ C_e) @ Pm
        exp_dec[k] = -1 if Xe[0] < 0 else 1

        # ---- fused filter ----
        C = np.array([[C_E], [c1h], [c2h]])
        R = np.diag([SIG_E2, max(s1h, EPS), max(s2h, EPS)])

        Xm = A @ Xf.reshape(-1, 1)
        Pm = A @ Pf @ A.T + Q
        resid = np.array([y_e[k], y1[k], y2[k]]) - (C @ Xm).flatten()
        K  = Pm @ C.T @ np.linalg.inv(C @ Pm @ C.T + R)
        Xf = (Xm + K @ resid.reshape(-1, 1)).flatten()
        Pf = (np.eye(1) - K @ C) @ Pm
        dec[k] = -1 if Xf[0] < 0 else 1

        # ---- parameter update, Eqs. (3.13)-(3.14) ----
        if learn:
            if   teacher_mode == "gt":     T = X[:k+1].astype(float)
            elif teacher_mode == "expert": T = exp_dec[:k+1]
            elif teacher_mode == "fused":  T = dec[:k+1]
            else: raise ValueError(teacher_mode)

            denom = T.copy()
            denom[np.abs(denom) < EPS] = EPS

            c1h = float(np.mean(y1[:k+1] / denom))
            s1h = float(np.var(y1[:k+1] - c1h * T))
            c2h = float(np.mean(y2[:k+1] / denom))
            s2h = float(np.var(y2[:k+1] - c2h * T))

        for key, val in zip(("c1", "c2", "s1", "s2"), (c1h, c2h, s1h, s2h)):
            hist[key][k] = val

    return dec, exp_dec, hist


def cov_XT(X_true, T):
    """Cov(X,T) = E[XT] - E[X]E[T].  T must be quantised to +-1.

    FIXED: the previous version returned only E[XT], which gave 1.0000
    for the annotated arm instead of 0.9313."""
    X_true = np.asarray(X_true, float)
    T = np.asarray(T, float)
    return float(np.mean(X_true * T) - np.mean(X_true) * np.mean(T))


## 3. Manuscript Table 4

Three checks are printed with the table:

* annotated Cov should be $4p(1-p) = 0.9313$
* expert Cov should be $0.2127$
* fused Cov should land in $[0.897,\ 0.911]$ — outside that range means
  the trajectory is not quantised


In [4]:
ARMS  = {"gt": "Annotated", "expert": "Expert teacher", "fused": "Fused"}
STYLE = {"gt": "seagreen",  "expert": "firebrick",      "fused": "steelblue"}

res = {}
expert_alone = np.where(y_e < 0, -1, 1)

print("=" * 70)
print(f"{'Trajectory':<16}{'Cov(X,T)':>10}{'Error':>9}{'c1':>9}{'c2':>9}"
      f"{'s1':>9}{'s2':>9}")
print("-" * 70)
print(f"{'Expert alone':<16}{'---':>10}{100*(expert_alone != X).mean():>8.2f}%"
      f"{'---':>9}{'---':>9}{'---':>9}{'---':>9}")

for mode, label in ARMS.items():
    dec, exp_dec, hist = run(mode)
    T = {"gt": X, "expert": exp_dec, "fused": dec}[mode]
    res[mode] = dict(dec=dec, hist=hist, cov=cov_XT(X, T),
                     err=100 * (dec != X).mean(),
                     cum=100 * np.cumsum(dec != X) / (np.arange(N) + 1))
    print(f"{label:<16}{res[mode]['cov']:>10.4f}{res[mode]['err']:>8.2f}%"
          f"{hist['c1'][-1]:>9.3f}{hist['c2'][-1]:>9.3f}"
          f"{hist['s1'][-1]:>9.3f}{hist['s2'][-1]:>9.3f}")

print("-" * 70)
print(f"references:              c1* = {C_REF[1]:.3f}  c2* = {C_REF[2]:.3f}"
      f"   s1* = {SIGMA_REF[1]:.3f}  s2* = {SIGMA_REF[2]:.3f}")
print("\nchecks:  annotated Cov = 0.9313 | expert Cov = 0.2127 | "
      "fused Cov in [0.897, 0.911]")
print("=" * 70)


Trajectory        Cov(X,T)    Error       c1       c2       s1       s2
----------------------------------------------------------------------
Expert alone           ---   28.47%      ---      ---      ---      ---


Annotated           0.9313    1.40%    4.035    5.923   11.448    7.535


Expert teacher      0.2127   24.20%    1.175    1.622   26.510   40.576


Fused               0.8995    1.35%    4.038    6.001   11.257    6.186
----------------------------------------------------------------------
references:              c1* = 4.332  c2* = 6.360   s1* = 11.366  s2* = 7.357

checks:  annotated Cov = 0.9313 | expert Cov = 0.2127 | fused Cov in [0.897, 0.911]


## 4. Figures

The couplings figure draws only the offline reference line. It does not
draw the Eq. (3.20) predicted limits — those belong to the bias analysis
in Chapter 3, and the chapter no longer re-applies them here.


In [5]:
# ---- Figure: cumulative error ----
fig, ax = plt.subplots(figsize=(9, 4.5))
ax.plot(100 * np.cumsum(expert_alone != X) / (np.arange(N) + 1),
        color="black", ls="--", lw=1.5,
        label=f"Expert sensor alone ({100*(expert_alone != X).mean():.2f}%)")
for mode in ("expert", "gt", "fused"):
    ax.plot(res[mode]["cum"], color=STYLE[mode], lw=1.8,
            label=f"{ARMS[mode]} ({res[mode]['err']:.2f}%)")
ax.set_xlabel("Time step $k$")
ax.set_ylabel("Cumulative classification error [%]")
ax.set_ylim(bottom=0)
ax.grid(alpha=0.3)
ax.legend(loc="upper right")
fig.tight_layout()
fig.savefig("teacher_error_binary.pdf")
plt.show()

# ---- Figure: learned couplings ----
fig, axes = plt.subplots(2, 1, sharex=True, figsize=(9, 7))
for idx, (ax, key) in enumerate(zip(axes, ("c1", "c2")), start=1):
    ref = C_REF[idx]
    ax.axhline(ref, color="black", ls="--", lw=1.4,
               label=f"Offline reference $c_{idx}^*$ = {ref:.3f}")
    for mode in ("expert", "gt", "fused"):
        ax.plot(res[mode]["hist"][key], color=STYLE[mode], lw=1.8,
                label=f"{ARMS[mode]} ({res[mode]['hist'][key][-1]:.3f})")
    ax.set_ylabel(rf"$\hat{{c}}_{idx}(k)$")
    ax.set_ylim(0, ref * 1.25)
    ax.grid(alpha=0.3)
    ax.legend(loc="lower right", fontsize=8, ncol=2)
axes[1].set_xlabel("Time step $k$")
fig.tight_layout()
fig.savefig("teacher_couplings_binary.pdf")
plt.show()

print("saved: teacher_error_binary.pdf, teacher_couplings_binary.pdf")


/tmp/ipykernel_873/341929382.py:16: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


saved: teacher_error_binary.pdf, teacher_couplings_binary.pdf


/tmp/ipykernel_873/341929382.py:34: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 5. Rescue rate (optional)

Section 4.2 of the chapter promises the rescue rate of Eq. (3.31) for
every arm, and no table currently reports it. Run this if you want to
restore that column; otherwise delete the promise from the chapter.

The rescue rate is the fraction of camera-wrong samples that the full
network nevertheless gets right — it isolates what the microphones
contribute, which aggregate error cannot.


In [6]:
wrong = (expert_alone != X)
n_wrong = int(wrong.sum())
print(f"camera wrong on {n_wrong}/{N} samples "
      f"({100*n_wrong/N:.2f}%)   Table 1 expects 527\n")
for mode, label in ARMS.items():
    rescued = ((res[mode]["dec"] == X) & wrong).sum()
    print(f"  {label:<16} rescue rate {100*rescued/n_wrong:6.2f}%")


camera wrong on 527/1851 samples (28.47%)   Table 1 expects 527

  Annotated        rescue rate  95.83%
  Expert teacher   rescue rate  14.99%
  Fused            rescue rate  96.02%
